In [10]:
# Test AWS connection

from pathlib import Path
from dotenv import load_dotenv
import os
import io
import boto3
import pandas as pd

PROJECT_DIR = Path.cwd().parent

load_dotenv(PROJECT_DIR / ".env")

AWS_REGION = os.getenv("AWS_REGION")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")

s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)

print("AWS region:", AWS_REGION)
print("S3 bucket:", S3_BUCKET_NAME)

AWS region: eu-central-1
S3 bucket: alex-jedha-kayak-data


In [2]:
s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)

response = s3.list_buckets()

print("Accessible S3 buckets:")

for bucket in response["Buckets"]:
    print("-", bucket["Name"])

Accessible S3 buckets:
- alex-jedha-kayak-data
- alexjedha-152727
- mlflow-final-project-artifacts
- mlflow-jedha-ab


In [5]:
# Upload your Kayak datasets to S3

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

files_to_upload = {

    RAW_DATA_DIR / "cities_coordinates.csv":
        "raw/cities_coordinates.csv",

    RAW_DATA_DIR / "weather_daily_7days.csv":
        "raw/weather_daily_7days.csv",

    RAW_DATA_DIR / "booking_data.csv":
        "raw/booking_data.csv",

    PROCESSED_DATA_DIR / "city_weather_ranking.csv":
        "processed/city_weather_ranking.csv",

    PROCESSED_DATA_DIR / "top_5_destinations.csv":
        "processed/top_5_destinations.csv",

    PROCESSED_DATA_DIR / "hotels_cleaned.csv":
        "processed/hotels_cleaned.csv",
}

In [6]:
# Check
for local_path, s3_key in files_to_upload.items():
    print(local_path.exists(), local_path.name, "->", s3_key)

True cities_coordinates.csv -> raw/cities_coordinates.csv
True weather_daily_7days.csv -> raw/weather_daily_7days.csv
True booking_data.csv -> raw/booking_data.csv
True city_weather_ranking.csv -> processed/city_weather_ranking.csv
True top_5_destinations.csv -> processed/top_5_destinations.csv
True hotels_cleaned.csv -> processed/hotels_cleaned.csv


In [7]:
# upload everything
for local_path, s3_key in files_to_upload.items():

    s3.upload_file(
        str(local_path),
        S3_BUCKET_NAME,
        s3_key
    )

    print(f"Uploaded: {s3_key}")

Uploaded: raw/cities_coordinates.csv
Uploaded: raw/weather_daily_7days.csv
Uploaded: raw/booking_data.csv
Uploaded: processed/city_weather_ranking.csv
Uploaded: processed/top_5_destinations.csv
Uploaded: processed/hotels_cleaned.csv


In [8]:
# Verify what is inside the bucket

response = s3.list_objects_v2(
    Bucket=S3_BUCKET_NAME
)

print("Files in S3:")

for obj in response.get("Contents", []):
    print("-", obj["Key"])

Files in S3:
- processed/city_weather_ranking.csv
- processed/hotels_cleaned.csv
- processed/top_5_destinations.csv
- raw/booking_data.csv
- raw/cities_coordinates.csv
- raw/weather_daily_7days.csv


Install the RDS/PostgreSQL libraries via terminal:

(base) PS C:\Users\Alex\Desktop\Jehda AI\Fullstack>
1. Go to folder: cd "CDSD Certification\1_Kayak"
2. Librarises: pip install sqlalchemy psycopg2-binary

3. Restart the Jupyter kernel


In [1]:
# Check environment
import sys
print(sys.version)

import sqlalchemy
print(sqlalchemy.__version__)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
2.0.52


In [ ]:
# Result: Python 3.13.9 is fine. SQLAlchemy has supported Python 3.13 since 2.0.31, 
# and the current 2.0.52 release supports it.

In [ ]:
#cHeck SQLAlchemy import
from sqlalchemy import create_engine, text

print("SQLAlchemy import successful")

SQLAlchemy import successful


In [ ]:
# after restarting the kernel, environment variables were cleared. I need to load .env again before 
# calling os.getenv().

from pathlib import Path
from dotenv import load_dotenv
import os

PROJECT_DIR = Path.cwd().parent

load_dotenv(PROJECT_DIR / ".env")

print("ENV loaded from:", PROJECT_DIR / ".env")

ENV loaded from: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\1_Kayak\.env


In [ ]:
# Load your RDS settings
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

RDS_HOST = os.getenv("RDS_HOST")
RDS_PORT = os.getenv("RDS_PORT")
RDS_DB = os.getenv("RDS_DB")
RDS_USER = os.getenv("RDS_USER")
RDS_PASSWORD = os.getenv("RDS_PASSWORD")

print("Host:", RDS_HOST)
print("Port:", RDS_PORT)
print("Database:", RDS_DB)
print("User:", RDS_USER)
print("Password loaded:", RDS_PASSWORD is not None)

Host: jedha-kayak-db.cvsi6keeephu.eu-central-1.rds.amazonaws.com
Port: 5432
Database: kayak_db
User: postgres
Password loaded: True


In [8]:
# Test the actual PostgreSQL connection


password_encoded = quote_plus(RDS_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{RDS_USER}:{password_encoded}"
    f"@{RDS_HOST}:{RDS_PORT}/{RDS_DB}"
)

engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT version();")
        )

        print("Connection successful!")
        print(result.fetchone()[0])

except Exception as e:
    print("Connection failed:")
    print(e)

Connection successful!
PostgreSQL 18.3 on x86_64-pc-linux-gnu, compiled by x86_64-pc-linux-gnu-gcc (GCC) 12.4.0, 64-bit


# ETL load: read the cleaned datasets from S3 and write them into PostgreSQL tables

In [11]:
# Read the cleaned hotel dataset from S3
obj = s3.get_object(
    Bucket=S3_BUCKET_NAME,
    Key="processed/hotels_cleaned.csv"
)

hotels_from_s3 = pd.read_csv(
    io.BytesIO(obj["Body"].read())
)

hotels_from_s3.head()

,hotel_id,city_id,name,rating,description,url,city,latitude,longitude
0,1,5,"Le Clémenceau - 6 pers, balcon & cœur de Rouen",NaN,Spacious Accommodations: Le Clémenceau in Roue...,https://www.booking.com/hotel/fr/le-clemenceau...,Rouen,49.434768,1.088898
1,2,5,ibis budget Rouen Centre Rive Gauche,8.3,Hotel ibis budget Rouen Center Rive Gauche Gau...,https://www.booking.com/hotel/fr/ibis-budget-r...,Rouen,49.425341,1.068718
2,3,5,"Radisson Blu Hotel, Rouen Centre",8.9,Comfortable Accommodations: Rooms feature air ...,https://www.booking.com/hotel/fr/radisson-blu-...,Rouen,49.446441,1.094120
3,4,5,Hyatt Place Rouen,8.6,Comfortable Accommodations: Hyatt Place Rouen ...,https://www.booking.com/hotel/fr/hyatt-place-r...,Rouen,49.453286,1.098285
4,5,5,B&B HOTEL Rouen Centre,6.8,"Ideally located for exploring Rouen, a city of...",https://www.booking.com/hotel/fr/comforthotelr...,Rouen,49.430326,1.084833


In [12]:
# Read the weather dataset from S3

obj = s3.get_object(
    Bucket=S3_BUCKET_NAME,
    Key="raw/weather_daily_7days.csv"
)

weather_from_s3 = pd.read_csv(
    io.BytesIO(obj["Body"].read())
)

weather_from_s3.head()

,city_id,city,latitude,longitude,date,temp_day,temp_min,temp_max,feels_like_day,humidity,weather,clouds,wind_speed,pop,rain
0,1,Mont Saint Michel,48.635954,-1.51146,2026-08-15,24.30,17.98,25.36,24.30,64,light rain,100,6.62,0.8,3.16
1,1,Mont Saint Michel,48.635954,-1.51146,2026-08-16,25.98,16.96,26.20,25.98,55,broken clouds,72,6.87,0.0,0.00
2,1,Mont Saint Michel,48.635954,-1.51146,2026-08-17,22.32,16.11,23.45,22.32,61,overcast clouds,97,5.92,0.0,0.00
3,1,Mont Saint Michel,48.635954,-1.51146,2026-08-18,22.72,16.53,24.38,22.72,66,broken clouds,75,6.08,0.0,0.00
4,1,Mont Saint Michel,48.635954,-1.51146,2026-08-19,20.18,16.18,21.96,20.18,62,overcast clouds,99,4.99,0.0,0.03


In [13]:
# Read Top 5 destinations from S3

obj = s3.get_object(
    Bucket=S3_BUCKET_NAME,
    Key="processed/top_5_destinations.csv"
)

top5_from_s3 = pd.read_csv(
    io.BytesIO(obj["Body"].read())
)

top5_from_s3.head()

,city_id,city,latitude,longitude,avg_temp_day,avg_temp_min,avg_temp_max,avg_humidity,avg_wind_speed,avg_pop,total_rain,temperature_score,rain_penalty,pop_penalty,wind_penalty,humidity_penalty,weather_score
0,5,Rouen,49.440459,1.093966,24.384286,15.421429,27.242857,45.428571,6.204286,0.197143,8.06,98.078571,16.12,19.714286,31.021429,21.857143,27.063571
1,6,Paris,48.853495,2.348391,25.417143,19.762857,29.638571,41.857143,4.685714,0.234286,5.19,92.914286,10.38,23.428571,23.428571,27.214286,26.181429
2,35,La Rochelle,46.159732,-1.151595,22.961429,18.761429,23.445714,62.714286,6.921429,0.291429,6.71,94.807143,13.42,29.142857,34.607143,4.071429,25.075000
3,7,Amiens,49.894171,2.295695,24.185714,15.728571,27.224286,44.428571,6.568571,0.271429,9.92,99.071429,19.84,27.142857,32.842857,23.357143,24.787857
4,1,Mont Saint Michel,48.635954,-1.511460,22.175714,16.080000,23.435714,60.428571,6.042857,0.171429,15.06,90.878571,30.12,17.142857,30.214286,0.642857,22.339286


In [ ]:
# Load 3 tables into PostgreSQL

hotels_from_s3.to_sql(
    "hotels",
    engine,
    if_exists="replace",
    index=False
)

weather_from_s3.to_sql(
    "weather",
    engine,
    if_exists="replace",
    index=False
)

top5_from_s3.to_sql(
    "top_5_destinations",
    engine,
    if_exists="replace",
    index=False
)

5

In [15]:
# Verify that the tables exist

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

pd.read_sql(query, engine)

,table_name
0,hotels
1,top_5_destinations
2,weather


In [16]:
# Test one SQL query

pd.read_sql(
    """
    SELECT
        hotel_id,
        city,
        name,
        rating
    FROM hotels
    ORDER BY rating DESC NULLS LAST
    LIMIT 20;
    """,
    engine
)

,hotel_id,city,name,rating
0,80,Amiens,l'esprit d'EMMA,9.8
1,94,Mont Saint Michel,Appart Standing - La Coque d'Or - Mont-St-Michel,9.6
2,79,Amiens,Elegant 120m2-Face cathedrale-6 pers-free Parking,9.5
3,13,Rouen,Le petit Rouen,9.5
4,11,Rouen,Le Majolique Nice and Cozy Apartment,9.4
5,10,Rouen,Duplex vieux marché quartier historique wifi/p...,9.4
6,15,Rouen,Joli appart cosy tout confort avec parking gra...,9.4
7,100,Mont Saint Michel,Résidence Beauvoir le Mont-Saint-Michel,9.4
8,19,Rouen,Le Petit Horloge,9.3
9,63,Amiens,Ginkgo Maison d'hôtes,9.3
